In [1]:
import warnings
import random
from typing import Iterable, Optional, Any

import numpy as np
import pandas as pd
import quantstats as qs
from sklearn.metrics import confusion_matrix, classification_report, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from src.helpers.model_matrix import build_model_matrix_from_wrds
from src.helpers.model_matrix import align_and_fill_dates_across_tickers
from src.helpers._extract import ensure_dir

warnings.filterwarnings(
    "ignore",
    message=r"pkg_resources is deprecated as an API\..*",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    message="Mean of empty slice",
    category=RuntimeWarning,
)

### 0. Trading Strategy (Long-Short Market Neutral) 

In [ ]:
# ranking helper
def ranking_equal_weights(scores, long_pct: float, short_pct: float) -> pd.Series:
    n = len(scores)
    n_long = max(1, int(n * long_pct))
    n_short = max(1, int(n * short_pct))
    idx = scores.sort_values(ascending=False).index
    w = pd.Series(0.0, index=scores.index)
    w.loc[idx[:n_long]] = 1.0 / n_long
    w.loc[idx[-n_short:]] = -1.0 / n_short
    return w


# volatility helpers
def rolling_annualized_vol(series: pd.Series, window: int = 20) -> pd.Series:
    # std of daily log-returns, then annualize
    vol = series.rolling(window=window, min_periods=max(1, window // 2)).std()
    return vol * np.sqrt(252)


def attach_volatility(
        frame: pd.DataFrame,
        ret_col: str = "adjclose_lead",
        idx_stock: str = "permno",
        idx_date: str = "date",
        window: int = 20,
        fill: str = "cross_section_median"
) -> pd.Series:
    # per-stock rolling vol
    vol = frame.groupby(level=idx_stock, group_keys=False)[ret_col].apply(
        lambda s: rolling_annualized_vol(s, window)
    )
    # cross-sectional fill
    if fill == "cross_section_median":
        vol = vol.groupby(level=idx_date).transform(lambda x: x.fillna(x.median()))
    return vol


# weighting transforms
def inverse_vol_adjust(raw_w: pd.Series, vol: pd.Series) -> pd.Series:
    return raw_w / vol


def normalize_dollar_neutral(w: pd.Series,
                             long_target: float = 0.5,
                             short_target: float = -0.5) -> pd.Series:
    pos = w > 0
    neg = w < 0
    long_sum = w[pos].sum()
    short_sum = -w[neg].sum()
    out = w.copy()
    if long_sum > 0:
        out[pos] = w[pos] / long_sum * long_target
    if short_sum > 0:
        out[neg] = w[neg] / short_sum * (-short_target)
    return out


# performance helpers
def compute_simple_returns(logret: pd.Series) -> pd.Series:
    return np.exp(logret) - 1.0


def portfolio_returns(weights: pd.Series,
                      logret: pd.Series,
                      idx_date: str = "date") -> pd.Series:
    # weights are per (permno, date); multiply and sum by date
    daily = (weights * logret).groupby(level=idx_date).sum()
    return daily


def equity_from_simple_returns(simple_daily: pd.Series) -> pd.Series:
    return (1 + simple_daily).cumprod()


def stats_from_returns(simple_daily: pd.Series) -> dict:
    mu = simple_daily.mean()
    sd = simple_daily.std(ddof=0)
    sharpe = (mu / sd) * np.sqrt(252) if sd > 0 else np.nan
    eq = equity_from_simple_returns(simple_daily)
    total_ret = (eq.iloc[-1] - 1) * 100
    max_dd = (eq / eq.cummax() - 1).min() * 100
    return dict(
        trading_days=len(simple_daily),
        total_return=total_ret,
        sharpe=sharpe,
        max_drawdown=max_dd,
        avg_daily_ret=mu * 100,
        daily_vol=sd * 100,
        equity=eq
    )

In [ ]:
def build_long_short_portfolio(
        frame: pd.DataFrame,
        score_col: str,
        ret_col: str = "adjclose_lead",
        idx_date: str = "date",
        idx_stock: str = "permno",
        long_pct: float = 0.20,
        short_pct: float = 0.20,
        vol_window: int = 20,
        long_target: float = 0.5,
        short_target: float = -0.5,
        example_date_idx: int | None = 0,
        label: str = "Portfolio"
):
    # 1) ranking (equal-weight within buckets) per date
    w_raw = frame.groupby(level=idx_date, group_keys=False)[score_col] \
        .apply(lambda s: ranking_equal_weights(s, long_pct, short_pct))

    # 2) attach volatility and inverse-vol adjust
    vol = attach_volatility(frame, ret_col=ret_col, idx_stock=idx_stock, idx_date=idx_date, window=vol_window)
    w_vol = inverse_vol_adjust(w_raw, vol)

    # 3) normalize to dollar neutral per date
    w = w_vol.groupby(level=idx_date, group_keys=False).apply(
        lambda g: normalize_dollar_neutral(g, long_target=long_target, short_target=short_target)
    )

    # 4) diagnostics (optional prints)
    print(f"\nStrategy: {label} (top/bottom {int(long_pct * 100)}%)")
    print(f"  Total positions (non-zero): {(w != 0).sum():,}")  # noqa
    print(f"  Long positions:             {(w > 0).sum():,}")
    print(f"  Short positions:            {(w < 0).sum():,}")

    # one-day example
    if example_date_idx is not None:
        udates = frame.index.get_level_values(idx_date).unique()
        if 0 <= example_date_idx < len(udates):
            d = udates[example_date_idx]
            mask_d = frame.index.get_level_values(idx_date) == d
            w_d = w.loc[mask_d]
            print(f"\nExample for {pd.Timestamp(d).date()}:")
            print(f"  # longs:  {(w_d > 0).sum()} | sum longs:  {w_d[w_d > 0].sum():.4f}")
            print(f"  # shorts: {(w_d < 0).sum()} | sum shorts: {w_d[w_d < 0].sum():.4f}")
            print(f"  Net exposure: {w_d.sum():.4f}  (target ~ 0.00)")
            print(f"  Gross exposure: {w_d.abs().sum():.4f} (target ~ 1.00)")

    # 5) returns/metrics
    simple_daily = portfolio_returns(w, frame[ret_col], idx_date=idx_date)
    simple_daily = compute_simple_returns(simple_daily.apply(np.log1p))

    stats = stats_from_returns(simple_daily)
    print("\nPerformance:")
    print(f"  Trading days:  {stats['trading_days']}")
    print(f"  Total Return:  {stats['total_return']:7.2f}%")
    print(f"  Sharpe Ratio:  {stats['sharpe']:7.2f}")
    print(f"  Max Drawdown:  {stats['max_drawdown']:7.2f}%")
    print(f"  Avg Daily Ret: {stats['avg_daily_ret']:7.4f}%")
    print(f"  Daily Vol:     {stats['daily_vol']:7.4f}%")

    return dict(weights=w, vol=vol, daily_ret=simple_daily, equity=stats["equity"], stats=stats)

### 1. Data build & cleaning (CRSP/DSF/IBES/FF)

In [2]:
tickers_list = [
    'AAPL', 'NVDA', 'MSFT', 'AMZN', 'TSLA',
    'GOOGL', 'LLY', 'WMT', 'JPM', 'BRK-B',
    'V', 'MA', 'XOM', 'ORCL', 'UNH', 'COST', 'PG', 'HD', 'NFLX',
    'JNJ', 'BAC', 'CRM', 'QQQ', 'ABBV', 'KO', 'CVX', 'TMUS', 'MRK', 'CSCO',
    'WFC', 'ACN', 'NOW', 'TSM', 'AXP', 'PEP', 'MCD', 'IBM', 'MS', 'DIS',
    'TMO', 'ABT', 'AMD', 'ADBE', 'PM', 'ISRG', 'GE', 'GS', 'INTU', 'CAT',
    'TXN', 'QCOM', 'RY', 'VZ', 'DHR', 'BKNG', 'T', 'BLK', 'SPGI',
    'RTX', 'PFE', 'NEE', 'HON', 'CMCSA', 'PGR', 'AMGN', 'LOW', 'ANET', 'UNP',
    'SYK', 'TJX', 'C', 'BA', 'SCHW', 'BSX', 'KKR', 'ETN',
    'COP', 'BX', 'PANW', 'ADP'
]

# will extract every possible ticker data (not just tickers_list) from wrds.
# the only wrds filters used are 'start' and 'end' dates.
# by extracting all possible ticker data, we can update tickers_list without
# connecting to wrds.
# the data are extracted from dsf, crsp, ff, and ibes (see src/migrations folder)
all_stocks = build_model_matrix_from_wrds(
    wrds_user="navid_namazi",
    start="2016-01-01",
    end="2021-01-01",
    chunk_size=500_000,
    tickers=tickers_list,
    use_run="last"  # "new", "last", or a specific folder name (i.e. "run_20250914_133747")
)

# ensure all stocks have the same date coverage
df = align_and_fill_dates_across_tickers(all_stocks=all_stocks)

[info] Using run folder: run_20250930_202918 (reuse=True)
[info] Reuse mode: all required Parquet files are present. No extraction performed.
{'run_folder': 'wrds_extracts/run_20250930_202918', 'reuse': True, 'artifacts': {'dsf.parquet': 'wrds_extracts/run_20250930_202918/dsf.parquet', 'stocknames.parquet': 'wrds_extracts/run_20250930_202918/stocknames.parquet', 'ff.parquet': 'wrds_extracts/run_20250930_202918/ff.parquet', 'ibes_stats.parquet': 'wrds_extracts/run_20250930_202918/ibes_stats.parquet', 'ibes_act.parquet': 'wrds_extracts/run_20250930_202918/ibes_act.parquet'}}
[info] Removed 0 permnos(companies) for having zero in cfacpr or cfacshr
[info] Removed 0 permnos(companies) for exceeding the threshold of negative prices
$$$$ df_prices initial shape :  (99461, 18)
[info] ibes_stats: 274,942 (official_ticker, stat_date) pairs have >1 row (multiple horizons/periodicities). Will collapse before join.
[info] df_prices(+ibes): 95.3% missing in n_analysts.
[info] df_prices(+ibes): 95.3%

### 2. Train/test split & rolling CV

In [3]:
random_state = 42
random.seed(random_state)

# get all unique dates
dates_all = df.index.get_level_values("date").unique().sort_values()
trading_days_count = len(dates_all)
n_rows = df.shape[0]

print("Total Data:")
print(f"  Dates: {trading_days_count} trading days")
print(f"  Period: {dates_all.min().date()} to {dates_all.max().date()}")
print(f"  Rows: {n_rows:,} (stocks × dates)")

# split: in-sample (80%) vs out-of-sample (20%)
split_pct = 0.80
split_idx = int(trading_days_count * split_pct)

# in-sample: used for feature selection, hyperparameter tuning, model development
ins_dates = dates_all[:split_idx]

# out-of-sample: never touched until final evaluation
dates_out_sample = dates_all[split_idx:]

# split date (boundary between in-sample and out-of-sample)
split_date = dates_out_sample[0]

print("\nData Split:")
print("   In-Sample (Development Set):")
print(f"   Period: {ins_dates.min().date()} to {ins_dates.max().date()}")
print(f"   Dates: {len(ins_dates)} days ({len(ins_dates) / trading_days_count * 100:.1f}%)")
print("   Purpose: feature selection, hyperparameter tuning, rolling CV")

print("\nOut-Of-Sample (Test Set):")
print(f"   Period: {dates_out_sample.min().date()} to {dates_out_sample.max().date()}")
print(f"   Dates: {len(dates_out_sample)} days ({len(dates_out_sample) / trading_days_count * 100:.1f}%)")
print("   Purpose: final performance evaluation only")

print(f"\nSplit Date: {split_date.date()}")

Total Data:
  Dates: 1184 trading days
  Period: 2016-04-20 to 2020-12-30
  Rows: 93,536 (stocks × dates)

Data Split:
   In-Sample (Development Set):
   Period: 2016-04-20 to 2020-01-23
   Dates: 947 days (80.0%)
   Purpose: feature selection, hyperparameter tuning, rolling CV

Out-Of-Sample (Test Set):
   Period: 2020-01-24 to 2020-12-30
   Dates: 237 days (20.0%)
   Purpose: final performance evaluation only

Split Date: 2020-01-24


In [4]:
# Rolling window size configuration for in-sample (60/20/20 Split)
# When naming variables, ins short for in-sample, oos short for out-of-sample 

ins_window_size = len(ins_dates)
ins_training_window_size = int(0.6 * ins_window_size)
ins_validation_window_size = int(0.2 * ins_window_size)
target_folds_count = 10

# Calculate step size to achieve target number of folds
# Formula: step = (remaining_data) / (target_folds - 1)
remaining_data = ins_window_size - ins_training_window_size - ins_validation_window_size
step_size = max(1, remaining_data // max(1, target_folds_count - 1))

# Calculate actual number of windows
actual_folds = (ins_window_size - ins_training_window_size - ins_validation_window_size) // step_size + 1

print("Rolling Window Configuration (In-Sample Only):")
print(
    f"   Training window: {ins_training_window_size} days (~{ins_training_window_size / 252:.1f} years, {ins_training_window_size / ins_window_size * 100:.1f}% of in-sample)")
print(
    f"   Validation window: {ins_validation_window_size} days (~{ins_validation_window_size / 252:.1f} years, {ins_validation_window_size / ins_window_size * 100:.1f}% of in-sample)")
print(f"   Step size: {step_size} days (~{step_size / 5:.1f} weeks)")
print(f"   Target folds: {target_folds_count}")
print(f"   Actual folds: {actual_folds}")
print(f"   Coverage: {actual_folds * ins_validation_window_size} validation days out of {ins_window_size} total")

Rolling Window Configuration (In-Sample Only):
   Training window: 568 days (~2.3 years, 60.0% of in-sample)
   Validation window: 189 days (~0.8 years, 20.0% of in-sample)
   Step size: 21 days (~4.2 weeks)
   Target folds: 10
   Actual folds: 10
   Coverage: 1890 validation days out of 947 total


### 3. Model Initial Setup: logistic regression + ElasticNet

In [5]:
# Binary Target Definition
print("Target Column: adjclose_lead = next-day log return(t -> t+1)")
print("We predict: will the stock go up (1) or down (0) tomorrow?")

DIR_binary = (df["adjclose_lead"] > 0).astype(int)  # 1 = up, 0 = down

# check class balance (market neutrality ~ 50/50)
print("\nBinary Target Distribution")
print(f" Up (1):   {(DIR_binary == 1).sum():,} observations ({(DIR_binary == 1).mean() * 100:.1f}%)")  # noqa
print(f" Down (0): {(DIR_binary == 0).sum():,} observations ({(DIR_binary == 0).mean() * 100:.1f}%)")  # noqa
print(f" Total:    {len(DIR_binary):,} observations")

# feature columns: everything except ticker, target, and the index columns, permno and date.
num_pred_cols = [c for c in df.columns if c not in (["ticker", "adjclose_lead"])]
print(f"\nUsing {len(num_pred_cols)} features for prediction")

Target Column: adjclose_lead = next-day log return(t -> t+1)
We predict: will the stock go up (1) or down (0) tomorrow?

Binary Target Distribution
 Up (1):   49,745 observations (53.2%)
 Down (0): 43,791 observations (46.8%)
 Total:    93,536 observations

Using 37 features for prediction


In [6]:
print("ElasticNet Hyperparameters and Pipeline Configuration")

# Hyperparameter Configuration: Using Defaults

# Moderate regularization
# Balances between preventing overfitting and retaining important features
ELASTICNET_C = 0.1

# 70% L1 / 30% L2
# L1 component does feature selection (zeros out weak features)
# L2 component handles multi-collinearity (keeps correlated features stable)
ELASTICNET_L1_RATIO = 0.7

print("\nHyperparameters:")
print(f"  C: {ELASTICNET_C} (moderate regularization)")
print(f"  l1_ratio: {ELASTICNET_L1_RATIO} (70% L1 / 30% L2)")

# Preprocessing: Standardize features
ct = ColumnTransformer([(
    "num",  # numerical
    # scales each feature so ElasticNet penalty treats them similarly (to avoid leakage).
    # scaling is done per fold-by-fold
    StandardScaler(with_mean=True),
    num_pred_cols  # only numerical columns
)],
    remainder="drop",  # columns not listed are dropped
    sparse_threshold=0.0  # force dense for feature importance
)

# Classifier with ElasticNet regularization
clf = LogisticRegression(
    penalty="elasticnet",  # Use ElasticNet (L1 + L2)
    solver="saga",  # Only solver supporting elasticnet
    l1_ratio=ELASTICNET_L1_RATIO,  # Mix of L1 and L2
    C=ELASTICNET_C,  # Regularization strength
    max_iter=5000,  # More iterations for convergence
    tol=1e-4,
    random_state=random_state,
    n_jobs=-1  # Use all CPUs
)

# Pipeline: preprocessing to classification
pipe = Pipeline([("prep", ct), ("clf", clf)])

print("\nPipeline Flow:")
print("   1) 'prep' = ColumnTransformer(ct): standardize features (fit on train only to avoid leakage")
print("   2) 'clf'  = LogisticRegression with ElasticNet: learns β per fold")

ElasticNet Hyperparameters and Pipeline Configuration

Hyperparameters:
  C: 0.1 (moderate regularization)
  l1_ratio: 0.7 (70% L1 / 30% L2)

Pipeline Flow:
   1) 'prep' = ColumnTransformer(ct): standardize features (fit on train only to avoid leakage
   2) 'clf'  = LogisticRegression with ElasticNet: learns β per fold


### 4. ElasticNet Hyperparameter tuning

In [18]:
# ElasticNet hyperparameter tuning (C, L1 ratio, class weight)

# set run_tuning=False to skip search and reuse previously chosen hyperparameters
RUN_TUNING = True

# small grid to keep runtime predictable
GRID_C = [0.05, 0.1, 0.2, 0.5]
GRID_L1_RATIO = [0.3, 0.5, 0.7, 0.9]
GRID_CLASS_WEIGHT: Optional[list[Optional[str]]] = None  # or [None, "balanced"]

# metrics
PRIMARY_METRIC = "mean_bal_acc"  # balanced accuracy across folds (primary)
TIEBREAKER_METRIC = "mean_sharpe"  # portfolio sharpe on validation windows (tie-breaker for top-n)

# early stop parameters to avoid unnecessary work
NO_IMPROVE_EPS = 0.002  # improvement threshold in balanced accuracy (0.2 pp)
NO_IMPROVE_K = 8  # stop if no improvement above threshold over last k evaluations
MIN_GAIN_EPS = 0.001  # if the gap between best and second-best is below this, call it plateau
HARD_CAP_M = 20  # hard cap on number of evaluated combinations
TIEBREAKER_TOP_N = 3  # compute portfolio tie-breaker only for top-n by primary metric

# portfolio tie-breaker settings (same construction as later in the notebook)
TB_LONG_PCT = 0.20
TB_SHORT_PCT = 0.20
TB_VOL_WINDOW = 20
TB_LONG_TRG = 0.5
TB_SHORT_TRG = -0.5


def _make_ct(feature_list: Iterable[str]) -> ColumnTransformer:
    # scale numeric features so the elasticnet penalty treats them comparably
    return ColumnTransformer(
        [("num", StandardScaler(with_mean=True), list(feature_list))],
        remainder="drop",
        sparse_threshold=0.0
    )


def _iter_rolling_windows(ins_dates, train_len, valid_len, step_size):
    # yield (train_dates, valid_dates) according to the existing rolling geometry
    start_pos = 0
    while start_pos + train_len + valid_len <= len(ins_dates):
        yield (
            ins_dates[start_pos: start_pos + train_len],
            ins_dates[start_pos + train_len: start_pos + train_len + valid_len],
        )
        start_pos += step_size


def _evaluation_on_fold(df, y, num_pred_cols, train_dates, valid_dates, params):
    # fit on the fold's training slice, evaluate on the fold's validation slice
    idx_tr = df.index.get_level_values("date").isin(train_dates)
    idx_va = df.index.get_level_values("date").isin(valid_dates)

    X_tr, y_tr = df.loc[idx_tr, num_pred_cols], y.loc[idx_tr]
    X_va, y_va = df.loc[idx_va, num_pred_cols], y.loc[idx_va]

    pipe = Pipeline([
        ("prep", _make_ct(num_pred_cols)),
        ("clf", LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            C=params["C"],
            l1_ratio=params["l1_ratio"],
            class_weight=params.get("class_weight"),
            max_iter=5000,
            tol=1e-4,
            random_state=42,
            n_jobs=-1
        ))
    ])
    pipe.fit(X_tr, y_tr)

    # primary metric: balanced accuracy on validation slice
    yhat_va = pipe.predict(X_va)
    bal_acc = balanced_accuracy_score(y_va, yhat_va)

    # tie-breaker metric: validation portfolio sharpe built from predicted scores
    p_up = pipe.predict_proba(X_va)[:, 1]
    score = 2 * p_up - 1
    tmp = df.loc[idx_va, ["adjclose_lead"]].copy()
    tmp["score"] = pd.Series(score, index=X_va.index)

    res = build_long_short_portfolio(
        frame=tmp,
        score_col="score",
        ret_col="adjclose_lead",
        idx_date="date",
        idx_stock="permno",
        long_pct=TB_LONG_PCT,
        short_pct=TB_SHORT_PCT,
        vol_window=TB_VOL_WINDOW,
        long_target=TB_LONG_TRG,
        short_target=TB_SHORT_TRG,
        example_date_idx=None,
        label="tune-valid"
    )
    sharpe = res["stats"].get("sharpe")

    return bal_acc, sharpe


def _combo_iterator(grid_c, grid_l1, grid_cw):
    # iterate grid in a coarse-to-fine order so good candidates appear early
    l1_order = [0.5, 0.7, 0.3, 0.9]
    l1_list = [x for x in l1_order if x in grid_l1] + [x for x in grid_l1 if x not in l1_order]
    if grid_cw is None:
        for c in sorted(grid_c):
            for l1 in l1_list:
                yield dict(C=c, l1_ratio=l1)
    else:
        for c in sorted(grid_c):
            for l1 in l1_list:
                for cw in grid_cw:
                    yield dict(C=c, l1_ratio=l1, class_weight=cw)


def tune_hyperparams(
        df, y, num_pred_cols, ins_dates, step_size,
        train_frac=0.60, valid_frac=0.20,
        no_improve_eps=NO_IMPROVE_EPS, no_improve_k=NO_IMPROVE_K,
        min_gain_eps=MIN_GAIN_EPS, hard_cap_m=HARD_CAP_M,
        tie_top_n=TIEBREAKER_TOP_N,
        grid_c=GRID_C, grid_l1=GRID_L1_RATIO, grid_cw=GRID_CLASS_WEIGHT
):
    # expose chosen params as module-level variables so later cells can reuse them
    global ELASTICNET_C, ELASTICNET_L1_RATIO, ELASTICNET_CLASS_WEIGHT

    # derive window lengths from the in-sample horizon
    ins_window_size = len(ins_dates)
    train_len = int(train_frac * ins_window_size)
    valid_len = int(valid_frac * ins_window_size)

    folds = list(_iter_rolling_windows(ins_dates, train_len, valid_len, step_size))
    print(f"[tune] using {len(folds)} rolling folds (train={train_len}, valid={valid_len})")

    results = []
    best_primary = -np.inf
    recent_primary = []  # tracks best-so-far to enforce the no-improvement rule
    evaluated = 0

    # grid sweep with early stopping
    for params in _combo_iterator(grid_c, grid_l1, grid_cw):
        evaluated += 1

        bal_accs = []
        for train_dates, valid_dates in folds:
            bal_acc, _ = _evaluation_on_fold(df, y, num_pred_cols, train_dates, valid_dates, params)
            bal_accs.append(bal_acc)

        mean_bal = float(np.mean(bal_accs))
        std_bal = float(np.std(bal_accs))
        results.append({**params, "mean_bal_acc": mean_bal, "std_bal_acc": std_bal})

        # update best-so-far
        if mean_bal > best_primary + 1e-12:
            best_primary = mean_bal
        recent_primary.append(best_primary)

        print(
            f"[tune] #{evaluated:02d} C={params.get('C'):.3f}, l1={params.get('l1_ratio'):.2f} "
            f"→ mean_bal_acc={mean_bal:.4f} | best_so_far={best_primary:.4f}"
        )

        # no-improvement stop: if best_so_far did not increase by at least epsilon over last k evaluations
        if len(recent_primary) >= no_improve_k:
            window_best = recent_primary[-1]
            window_prev = recent_primary[-no_improve_k]
            if (window_best - window_prev) < no_improve_eps:
                print(f"[tune] early stop: no improvement ≥ {no_improve_eps} over last {no_improve_k} combos.")
                break

        # hard cap stop: absolute limit on number of evaluated combinations
        if evaluated >= hard_cap_m:
            print(f"[tune] early stop: hard cap m={hard_cap_m}.")
            break

    # aggregate results and pick the primary winner
    res_df = pd.DataFrame(results).sort_values("mean_bal_acc", ascending=False).reset_index(drop=True)

    # minimal-gain note: if best and second-best are very close, highlight plateau
    if len(res_df) >= 2:
        gap = res_df.loc[0, "mean_bal_acc"] - res_df.loc[1, "mean_bal_acc"]
        if gap < min_gain_eps:
            print(f"[tune] minimal gain achieved (Δ={gap:.4f} < {min_gain_eps}). further tuning likely not useful.")

    # portfolio tie-breaker for the top-n by balanced accuracy
    if len(res_df) > 1:
        n_tb = min(tie_top_n, len(res_df))
        print(f"[tune] computing tie-breaker sharpe for top {n_tb} candidates...")
        for i in range(n_tb):
            row = res_df.iloc[i].to_dict()
            params_tb = {k: row[k] for k in ["C", "l1_ratio"] if k in row}
            sharpes = []
            for train_dates, valid_dates in folds:
                _, sh = _evaluation_on_fold(df, y, num_pred_cols, train_dates, valid_dates, params_tb)
                if sh is not None:
                    sharpes.append(sh)
            if sharpes:
                res_df.loc[i, "mean_sharpe"] = float(np.mean(sharpes))
                res_df.loc[i, "std_sharpe"] = float(np.std(sharpes))

        # re-rank by primary metric then sharpe as tie-breaker
        res_df = res_df.sort_values(["mean_bal_acc", "mean_sharpe"], ascending=[False, False]).reset_index(drop=True)

    # finalize chosen hyperparameters
    best_row = res_df.iloc[0]
    ELASTICNET_C = best_row["C"]
    ELASTICNET_L1_RATIO = best_row["l1_ratio"]
    ELASTICNET_CLASS_WEIGHT = best_row.get("class_weight", None)

    print("[tune] best params:")
    print(
        f"   C={ELASTICNET_C}, l1_ratio={ELASTICNET_L1_RATIO}"
        + (f", class_weight={ELASTICNET_CLASS_WEIGHT}" if ELASTICNET_CLASS_WEIGHT is not None else "")
    )
    print(
        f"   mean_bal_acc={best_row['mean_bal_acc']:.4f}"
        + (f", mean_sharpe={best_row.get('mean_sharpe', np.nan):.2f}" if 'mean_sharpe' in best_row else "")
    )

    return dict(C=ELASTICNET_C, l1_ratio=ELASTICNET_L1_RATIO, class_weight=ELASTICNET_CLASS_WEIGHT), res_df


# run or reuse tuning
if RUN_TUNING:
    best_params, tuning_results = tune_hyperparams(
        df=df, y=DIR_binary, num_pred_cols=num_pred_cols,
        ins_dates=ins_dates, step_size=step_size
    )
    print("top 5 candidates:")
    print(tuning_results.head(5).to_string(index=False))
else:
    # set from a previous run or manual choice
    ELASTICNET_C = 0.1
    ELASTICNET_L1_RATIO = 0.7
    ELASTICNET_CLASS_WEIGHT = None
    print(
        f"[tune] skipped tuning. using predefined params: "
        f"C={ELASTICNET_C}, l1_ratio={ELASTICNET_L1_RATIO}, class_weight={ELASTICNET_CLASS_WEIGHT}"
    )

[tune] Using 10 rolling folds (train=568, valid=189, step=21)

[tune] Eval #1 → C=0.05, l1_ratio=0.5

Strategy: tune-valid (top/bottom 20%)
  Total positions (non-zero): 6,111
  Long positions:             2,700
  Short positions:            2,700

Performance:
  Trading days:  189
  Total Return:     4.85%
  Sharpe Ratio:     1.03
  Max Drawdown:    -3.61%
  Avg Daily Ret:  0.0259%
  Daily Vol:      0.3969%

Strategy: tune-valid (top/bottom 20%)
  Total positions (non-zero): 6,111
  Long positions:             2,700
  Short positions:            2,700

Performance:
  Trading days:  189
  Total Return:     1.95%
  Sharpe Ratio:     0.44
  Max Drawdown:    -4.50%
  Avg Daily Ret:  0.0110%
  Daily Vol:      0.3962%

Strategy: tune-valid (top/bottom 20%)
  Total positions (non-zero): 6,111
  Long positions:             2,700
  Short positions:            2,700

Performance:
  Trading days:  189
  Total Return:     5.01%
  Sharpe Ratio:     1.03
  Max Drawdown:    -3.80%
  Avg Daily Ret:  

KeyboardInterrupt: 

### 5. Rolling training & Feature selection per fold

In [7]:
print("Rolling Window Training With Feature Selection (In-Sample Only)")

pred_prob_up_new = pd.Series(index=df.index, dtype=float)
pred_prob_down_new = pd.Series(index=df.index, dtype=float)
pred_score_new = pd.Series(index=df.index, dtype=float)
pred_class_new = pd.Series(index=df.index, dtype=int)
used_mask_new = pd.Series(False, index=df.index)

# track feature selection across windows
feature_selection_history = []

window_num = 0
start_pos = 0

while start_pos + ins_training_window_size + ins_validation_window_size <= len(ins_dates):
    window_num += 1

    train_dates = ins_dates[start_pos: start_pos + ins_training_window_size]
    valid_dates = ins_dates[
                  start_pos + ins_training_window_size: start_pos + ins_training_window_size + ins_validation_window_size]

    print(f"\nWindow {window_num}: {train_dates.min().date()} to {valid_dates.max().date()}")

    start_pos += step_size

    idx_tr = df.index.get_level_values("date").isin(train_dates)
    idx_va = df.index.get_level_values("date").isin(valid_dates)

    X_tr = df.loc[idx_tr, num_pred_cols]
    y_tr = DIR_binary.loc[idx_tr]
    X_va = df.loc[idx_va, num_pred_cols]
    y_va = DIR_binary.loc[idx_va]

    # train model
    pipe.fit(X_tr, y_tr)

    # feature Selection Analysis
    coef = pipe.named_steps["clf"].coef_[0]

    # features with non-zero coefficients
    feature_importance = pd.DataFrame({
        "feature": num_pred_cols,
        "coefficient": coef,
        "abs_coefficient": np.abs(coef)
    })

    # count selected features (abs(coef) > threshold)
    ZERO_THRESHOLD = 1e-5
    selected_features = feature_importance[feature_importance["abs_coefficient"] > ZERO_THRESHOLD]
    n_selected = len(selected_features)
    pct_selected = (n_selected / len(num_pred_cols)) * 100

    # top 10 most important features
    top_features = selected_features.nlargest(10, "abs_coefficient")

    # store feature selection info
    feature_selection_history.append({
        "window": window_num,
        "n_features_selected": n_selected,
        "pct_selected": pct_selected,
        "selected_features": selected_features["feature"].tolist(),  # all selected features
        "top_5_features": top_features.head(5)["feature"].tolist(),
        "train_start": train_dates.min(),
        "valid_end": valid_dates.max()
    })

    # predict
    proba = pipe.predict_proba(X_va)
    p_down = proba[:, 0]
    p_up = proba[:, 1]
    score = 2 * p_up - 1
    yhat = (p_up > 0.5).astype(int)

    # store predictions
    pred_prob_up_new.loc[idx_va] = p_up
    pred_prob_down_new.loc[idx_va] = p_down
    pred_score_new.loc[idx_va] = score
    pred_class_new.loc[idx_va] = yhat
    used_mask_new.loc[idx_va] = True

    # report
    accuracy = (yhat == y_va).mean()  # noqa
    print(f"Training: {len(X_tr):,} samples")
    print(f"Validation: {len(X_va):,} samples, Accuracy: {accuracy:.1%}")
    print("\nFeature Selection:")
    print(f"   Selected: {n_selected}/{len(num_pred_cols)} features ({pct_selected:.1f}%)")
    print("   Top 5 features by importance:")
    for i, row in top_features.head(5).iterrows():
        print(f"      {i + 1}. {row['feature']}: {row['coefficient']:+.4f}")

print(f"\nTraining complete: {window_num} windows processed")
print(f"Total validated: {used_mask_new.sum():,} / {len(df):,}")

# update global predictions with new ones
pred_prob_up = pred_prob_up_new
pred_prob_down = pred_prob_down_new
pred_score = pred_score_new
pred_class = pred_class_new
used_mask = used_mask_new

Rolling Window Training With Feature Selection (In-Sample Only)

Window 1: 2016-04-20 to 2019-04-23
Training: 44,872 samples
Validation: 14,931 samples, Accuracy: 52.1%

Feature Selection:
   Selected: 27/37 features (73.0%)
   Top 5 features by importance:
      22. ti_rsi_28: +0.0886
      21. ti_rsi_14: -0.0754
      17. smb: +0.0694
      3. retx: -0.0683
      16. mktrf: -0.0471

Window 2: 2016-05-19 to 2019-05-22
Training: 44,872 samples
Validation: 14,931 samples, Accuracy: 52.1%

Feature Selection:
   Selected: 26/37 features (70.3%)
   Top 5 features by importance:
      17. smb: +0.0677
      3. retx: -0.0560
      16. mktrf: -0.0446
      34. ti_stoch_k_14_3_3: +0.0315
      22. ti_rsi_28: +0.0270

Window 3: 2016-06-20 to 2019-06-21
Training: 44,872 samples
Validation: 14,931 samples, Accuracy: 51.5%

Feature Selection:
   Selected: 28/37 features (75.7%)
   Top 5 features by importance:
      17. smb: +0.0641
      16. mktrf: -0.0558
      3. retx: -0.0558
      34. ti_stoc

### 6. Feature selection across all folds

In [8]:
print("Feature Selection For Final Model")

# extract selection information from all windows
n_windows = len(feature_selection_history)
feature_selected_counts = {feat: 0 for feat in num_pred_cols}

# count how many windows each feature was selected in
for window_info in feature_selection_history:
    selected_features = window_info["selected_features"]
    for feat in selected_features:
        feature_selected_counts[feat] += 1

# calculate frequency (percentage of windows)
feature_freq = pd.DataFrame({
    "feature": list(feature_selected_counts.keys()),
    "count": list(feature_selected_counts.values()),
    "frequency": [count / n_windows for count in feature_selected_counts.values()]
}).sort_values("frequency", ascending=False)

SELECTION_THRESHOLD = 0.5  # moderate

selected_features_mask = feature_freq["frequency"] >= SELECTION_THRESHOLD
final_feature_list = feature_freq.loc[selected_features_mask, "feature"].tolist()

print("\n  Selection Criterion:")
print(f"   Frequency threshold: ≥ {SELECTION_THRESHOLD * 100:.0f}% of windows")
print("   Rationale: Features must be consistently predictive across")
print("              different market regimes to be included")

print("\n  Results:")
print(f"   Features selected: {len(final_feature_list)} / {len(num_pred_cols)}")
print(f"   Features removed:  {len(num_pred_cols) - len(final_feature_list)}")
print(f"   Reduction: {(1 - len(final_feature_list) / len(num_pred_cols)) * 100:.1f}%")

# display selected features
print(f"\n  Selected Features ({len(final_feature_list)}):")
print(f"    {'Feature':<30} {'Frequency':<12}     {'Appearances':<15}")

selected_features_df = feature_freq[selected_features_mask].copy()
for idx, row in selected_features_df.iterrows():
    feat = row['feature']
    freq = row['frequency']
    count = row['count']
    bar = '█' * int(freq * 10)
    print(f"    {feat:<30} {freq:>6.1%} ({count:>2}/{n_windows})   {bar}")

# display removed features
removed_features_df = feature_freq[~selected_features_mask].copy()

if len(removed_features_df) > 0:
    print(f"\n  Removed Features ({len(removed_features_df)}):")
    print(f"    {'Feature':<30} {'Frequency':<12}     {'Appearances':<15}")

    for idx, row in removed_features_df.iterrows():
        feat = row['feature']
        freq = row['frequency']
        count = row['count']
        bar = '░' * int(freq * 10)
        print(f"    {feat:<30} {freq:>6.1%} ({count:>2}/{n_windows})   {bar}")


# feature categories breakdown
def categorize_feature(feat):
    """Categorize feature by type"""
    if feat.startswith("ti_"):
        return "Technical Indicators"
    elif feat in ["mktrf", "smb", "hml", "rf", "umd"]:
        return "Fama-French Factors"
    elif feat.startswith("cons_") or feat.startswith("n_"):
        return "IBES Consensus"
    elif feat.startswith("adjclose_lag"):
        return "Price Lags"
    elif feat in ["adj_mktcap", "vol", "retx"]:
        return "Market Data"
    else:
        return "Other"


selected_features_df["category"] = selected_features_df["feature"].apply(categorize_feature)
category_counts = selected_features_df["category"].value_counts()

print("\n  Selected Features By Category:")

for category, count in category_counts.items():
    pct = count / len(selected_features_df) * 100
    print(f"    {category:<30} {count:>2} features ({pct:>5.1f}%)")

Feature Selection For Final Model

  Selection Criterion:
   Frequency threshold: ≥ 50% of windows
   Rationale: Features must be consistently predictive across
              different market regimes to be included

  Results:
   Features selected: 29 / 37
   Features removed:  8
   Reduction: 21.6%

  Selected Features (29):
    Feature                        Frequency        Appearances    
    rf                             100.0% (10/10)   ██████████
    ti_MACD_12_26_9                100.0% (10/10)   ██████████
    ti_mfi_14                      100.0% (10/10)   ██████████
    ti_adx_14                      100.0% (10/10)   ██████████
    umd                            100.0% (10/10)   ██████████
    vol                            100.0% (10/10)   ██████████
    hml                            100.0% (10/10)   ██████████
    smb                            100.0% (10/10)   ██████████
    mktrf                          100.0% (10/10)   ██████████
    adjclose_lag3                  10

### 7. Train the final in-sample model on all in-sample data

In [9]:
print("\nTraining Final Model On All In-Sample Data")

# Rebuild Pipeline With Selected Features Only
ct_final = ColumnTransformer([
    ("num", StandardScaler(with_mean=False), final_feature_list)
], remainder="drop")

clf_final = clf  # reuse same model as before 
pipe_final = Pipeline([("prep", ct_final), ("clf", clf_final)])

print(f"\n Pipeline rebuilt with {len(final_feature_list)} selected features")
print(f"  (Original pipeline used all {len(num_pred_cols)} features)")

# train On All In-Sample Data
ins_mask = df.index.get_level_values("date").isin(ins_dates)
X_train_final = df.loc[ins_mask, final_feature_list]
y_train_final = DIR_binary.loc[ins_mask]

pipe_final.fit(X_train_final, y_train_final)

# Analyze final model
# check which of the selected features ElasticNet actually uses
final_coef = pipe_final.named_steps["clf"].coef_[0]
non_zero_mask = np.abs(final_coef) > 1e-5
actually_used_features = np.array(final_feature_list)[non_zero_mask]

print("\nFinal model trained on:")
print(f"  Period: {ins_dates.min().date()} to {ins_dates.max().date()}")
print(f"  Days: {len(ins_dates)}")
print(f"  Samples: {len(X_train_final):,}")

print("\nFeature Selection (2-stage):")
print(f"  Stage 1 (Frequency-based): {len(final_feature_list)} / {len(num_pred_cols)} features selected")
print(f"  Stage 2 (ElasticNet on final model): {len(actually_used_features)} / {len(final_feature_list)} features used")
print(
    f"  Total reduction: {len(num_pred_cols)} -> {len(actually_used_features)} ({len(actually_used_features) / len(num_pred_cols) * 100:.1f}%)")

print("\nTop 10 features by coefficient magnitude:")
feature_importance = pd.DataFrame({
    "feature": final_feature_list,
    "coefficient": final_coef,
    "abs_coefficient": np.abs(final_coef)
}).sort_values("abs_coefficient", ascending=False)

for i, (idx, row) in enumerate(feature_importance.head(10).iterrows(), 1):
    feat = row["feature"]
    coef = row["coefficient"]
    print(f"  {i:2d}. {feat:25s} {coef:+.4f}")


Training Final Model On All In-Sample Data

 Pipeline rebuilt with 29 selected features
  (Original pipeline used all 37 features)

Final model trained on:
  Period: 2016-04-20 to 2020-01-23
  Days: 947
  Samples: 74,813

Feature Selection (2-stage):
  Stage 1 (Frequency-based): 29 / 37 features selected
  Stage 2 (ElasticNet on final model): 26 / 29 features used
  Total reduction: 37 -> 26 (70.3%)

Top 10 features by coefficient magnitude:
   1. smb                       +0.0469
   2. ti_stoch_k_14_3_3         +0.0436
   3. retx                      -0.0428
   4. ti_rsi_28                 +0.0392
   5. vol                       -0.0374
   6. ti_rsi_14                 -0.0316
   7. cons_range_pct            -0.0253
   8. ti_aroon_osc_25           -0.0205
   9. ti_kurtosis_63            +0.0202
  10. rf                        +0.0187


### 8. In-sample portfolio construction (Ranking-based trading strategy)

In [12]:
ins_df = df.loc[used_mask].copy()
ins_df["score"] = pred_score.loc[used_mask]

ins_result = build_long_short_portfolio(
    frame=ins_df,
    score_col="score",
    ret_col="adjclose_lead",
    idx_date="date",
    idx_stock="permno",
    long_pct=0.20,
    short_pct=0.20,
    vol_window=20,
    long_target=0.5,
    short_target=-0.5,
    example_date_idx=10,
    label="In-sample (validation windows only)"
)


Strategy: In-sample (validation windows only) (top/bottom 20%)
  Total positions (non-zero): 11,781
  Long positions:             5,535
  Short positions:            5,535

Example for 2018-08-06:
  # longs:  15 | sum longs:  0.5000
  # shorts: 15 | sum shorts: -0.5000
  Net exposure: 0.0000  (target ~ 0.00)
  Gross exposure: 1.0000 (target ~ 1.00)

Performance:
  Trading days:  378
  Total Return:    -1.23%
  Sharpe Ratio:    -0.14
  Max Drawdown:    -6.11%
  Avg Daily Ret: -0.0028%
  Daily Vol:      0.3174%


### 9. Out-of-sample evaluation & portfolio construction

In [16]:
print("Out-Of-Sample Evaluation")

# define oos slice
oos_mask = df.index.get_level_values("date").isin(dates_out_sample)

# X_test: the features selected by the stability+ElasticNet process
# y_test: the binary target (Up=1, Down=0)
X_test = df.loc[oos_mask, final_feature_list]
y_test = DIR_binary.loc[oos_mask]

print("\nOut-of-sample period:")
print(f"  Dates: {dates_out_sample.min().date()} to {dates_out_sample.max().date()}")
print(f"  Days: {len(dates_out_sample)}")
print(f"  Samples: {len(X_test):,}")

# score with the frozen final model
# - predict_proba: probability of Up tomorrow
# - score: signed score in [-1, +1] (centered probability)
# - predict: class label (0/1)
test_prob_up = pipe_final.predict_proba(X_test)[:, 1]
test_prob_down = 1 - test_prob_up
test_score = 2 * test_prob_up - 1
test_pred = pipe_final.predict(X_test)

# classification summary
test_accuracy = (test_pred == y_test).mean()  # noqa

print("\nClassification Performance:")
print(f"  Accuracy: {test_accuracy:.1%}")
print(f"  Up class accuracy: {(test_pred[y_test == 1] == 1).mean():.1%}")  # noqa
print(f"  Down class accuracy: {(test_pred[y_test == 0] == 0).mean():.1%}")  # noqa

# confusion matrix + detailed classification report
cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
cm_df = pd.DataFrame(
    cm, index=["Actual Down (0)", "Actual Up (1)"],
    columns=["Pred Down (0)", "Pred Up (1)"]
)
print("\nConfusion Matrix:")
print(cm_df)

report = classification_report(
    y_test,
    test_pred,
    labels=[0, 1],
    target_names=["Down (0)", "Up (1)"],
    zero_division=0
)
print("\nClassification Report")
print(report)

pred_prob_up_test = pd.Series(test_prob_up, index=X_test.index, name="prob_up")
pred_score_test = pd.Series(test_score, index=X_test.index, name="score")
pred_class_test = pd.Series(test_pred, index=X_test.index, name="pred_class")

print(f"\nTest predictions created for {len(pred_score_test):,} observations")
print("\nBuilding OOS Long-Short Portfolio (Top/Bottom 20%)")

oos_df = df.loc[oos_mask].copy()
oos_df["score"] = pred_score_test
oos_df["pred_class"] = pred_class_test
oos_df["prob_up"] = pred_prob_up_test

oos_result = build_long_short_portfolio(
    frame=oos_df,
    score_col="score",
    ret_col="adjclose_lead",
    idx_date="date",
    idx_stock="permno",
    long_pct=0.20,
    short_pct=0.20,
    vol_window=20,
    long_target=0.5,
    short_target=-0.5,
    example_date_idx=10,
    label="Out-of-sample"
)

stats = oos_result["stats"]
print("\nOOS Long-Short Headline Stats:")
print(f"  Total Return:   {stats.get('total_return', float('nan')):.2f}%")
print(f"  Sharpe Ratio:   {stats.get('sharpe', float('nan')):.2f}")
print(f"  Max Drawdown:   {stats.get('max_drawdown', float('nan')):.2f}%")
print(f"  Avg Daily Ret:  {stats.get('avg_daily_ret', float('nan')):.4f}%")
print(f"  Daily Vol:      {stats.get('daily_vol', float('nan')):.4f}%")

Out-Of-Sample Evaluation

Out-of-sample period:
  Dates: 2020-01-24 to 2020-12-30
  Days: 237
  Samples: 18,723

Classification Performance:
  Accuracy: 51.6%
  Up class accuracy: 80.5%
  Down class accuracy: 19.9%

Confusion Matrix:
                 Pred Down (0)  Pred Up (1)
Actual Down (0)           1780         7153
Actual Up (1)             1912         7878

Classification Report
              precision    recall  f1-score   support

    Down (0)       0.48      0.20      0.28      8933
      Up (1)       0.52      0.80      0.63      9790

    accuracy                           0.52     18723
   macro avg       0.50      0.50      0.46     18723
weighted avg       0.50      0.52      0.47     18723


Test predictions created for 18,723 observations

Building OOS Long-Short Portfolio (Top/Bottom 20%)

Strategy: Out-of-sample (top/bottom 20%)
  Total positions (non-zero): 7,551
  Long positions:             3,420
  Short positions:            3,420

Example for 2020-02-07:
  # lon

### 10. Generating Out-Of-Sample Reports

In [17]:
print("Generating Out-Of-Sample HTML Reports")


def make_qs_report_from_equity(equity_series, rf_series, mktrf_series, title, out_path):
    # Daily simple returns from equity
    rets = equity_series.pct_change().dropna()
    # Align RF & benchmark on the same index
    rf_aligned = rf_series.reindex(rets.index).ffill().bfill()
    mktrf_aln = mktrf_series.reindex(rets.index).ffill().bfill()
    bench_simple = (mktrf_aln + rf_aligned)
    # Excess returns
    strat_excess = (rets - rf_aligned).dropna()
    bench_excess = (bench_simple - rf_aligned).reindex(strat_excess.index).dropna()
    # Align both
    common_idx = strat_excess.index.intersection(bench_excess.index)
    strat_excess = strat_excess.reindex(common_idx)
    bench_excess = bench_excess.reindex(common_idx)

    qs.reports.html(
        strat_excess,
        benchmark=bench_excess.to_frame("Market"),
        rf=0.0,
        periods_per_year=252,
        output=out_path,
        title=title
    )
    print(f"    Saved: {out_path}")
    print(f"   Period: {strat_excess.index.min().date()} to {strat_excess.index.max().date()}")
    print(f"   Days:   {len(strat_excess)}")


# Prepare OOS Fama-French series
used_mask_oos = oos_result["weights"] != 0  # weights inside oos_result
rf_oos = (
    oos_df.loc[used_mask_oos]
    .reset_index()[["date", "rf"]]
    .dropna()
    .groupby("date", as_index=True)["rf"]
    .mean()
    .astype(float)
    .sort_index()
)
mktrf_oos = (
    oos_df.loc[used_mask_oos]
    .reset_index()[["date", "mktrf"]]
    .dropna()
    .groupby("date", as_index=True)["mktrf"]
    .mean()
    .astype(float)
    .sort_index()
)

# Generate HTML Reports: Out-Of-Sample Only (Both Strategies)
ensure_dir("out")
make_qs_report_from_equity(
    equity_series=oos_result["equity"],
    rf_series=rf_oos,
    mktrf_series=mktrf_oos,
    title=f"OUT-OF-SAMPLE: Long-Short Market Neutral ({dates_out_sample.min().date()} to {dates_out_sample.max().date()})",
    out_path="out/oos_long_short_tearsheet.html"
)

Generating Out-Of-Sample HTML Reports
    Saved: out/oos_long_short_tearsheet.html
   Period: 2020-01-27 to 2020-12-30
   Days:   236
